# NB07 – Cross-Border-Analyse: CH Import/Export & Grenzflüsse
### CAS Information Engineering – Scripting Project (Kür)
**Gruppe:** SC26_Gruppe_2 | **Datum:** März–Mai 2026

---
Analysiert die physikalischen Grenzflüsse der Schweiz mit DE, AT, IT und FR.  
Verknüpft Import/Export-Muster mit Spot-Preisen — empirische Validierung des Business Case.

**Aufbau:** Daten laden → Analyse → Visualisierung → Fazit

---
| [← NB06 Räumliche Analyse](06_Raeumliche_Analyse.ipynb) | [↑ Projektübersicht](00_Project_Overview.ipynb) | [→ NB08 Marktdynamik](08_Marktdynamik.ipynb) |
|:---|:---:|---:|


---

## 1. Daten laden

Kür-Notebooks laden ihre Datensätze im ersten Abschnitt des Notebooks,
das sie erstmals benötigt — hier NB07 für die ENTSO-E Grenzflüsse
und die daraus abgeleitete Import/Export-Analyse.

### Datensätze dieses Notebooks

| Datei | Quelle | Pfad | Erstellt in |
|-------|--------|------|-------------|
| `ch_crossborder_raw.csv` | ENTSO-E `query_crossborder_flows` | `data/raw/` | **NB07 (dieses NB)** |
| `ch_import_export_analyse.csv` | Join Grenzflüsse + Spot-Preise | `data/intermediate/` | **NB07 (dieses NB)** |

> `force_reload.crossborder: true` in `config.json` um Rohdaten neu zu laden.
> `force_reload.import_export: true` um Analyse neu zu berechnen.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import os, warnings, json as _json
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')

with open('config.json') as _f:
    CFG = _json.load(_f)

MODE          = CFG['mode']
DIR_RAW       = os.path.join('data', 'raw')
DIR_PROCESSED = os.path.join('data', 'processed')
DIR_INTER     = os.path.join('data', 'intermediate')
SZ_AKTIV      = CFG['szenarien']['gleichzeitigkeit_aktiv']
DIR_INTER_SZ  = os.path.join(DIR_INTER, SZ_AKTIV)
CHARTS_DIR       = os.path.join('output', 'charts', SZ_AKTIV)
os.makedirs(CHARTS_DIR, exist_ok=True)
DPI = CFG['visualisierung']['output_dpi']  # SSOT: config.json

_viz=CFG.get('visualisierung',{}).get('farben',{})
BG_DARK  = _viz.get('bg_dark',  '#0d1117')
BG_PANEL = _viz.get('bg_panel', '#141414')
C_PRICE  = _viz.get('c_price',  '#FFA726')
C_LOAD   = _viz.get('c_load',   '#66BB6A')
C_CHARGE = _viz.get('c_charge', '#1565C0')
C_FEED   = _viz.get('c_feed',   '#B71C1C')
SEG_COLORS = _viz.get('seg_colors', ['#42A5F5','#66BB6A','#FFA726','#EF5350'])

print(f'MODE         : {MODE}')
print(f'Szenario     : {SZ_AKTIV}')
print(f'intermediate : {os.path.abspath(DIR_INTER_SZ)}')
print(f'output       : {os.path.abspath(CHARTS_DIR)}')

import datetime as _dt

def log_dataindex(filename, source_url, local_path, data_type,
                  rows=None, size_kb=None, status='active', note=''):
    """Registriert Datei in dataindex.csv (append-only)."""
    DATAINDEX = 'dataindex.csv'
    import pandas as pd
    ts = _dt.datetime.utcnow().isoformat(timespec='seconds') + 'Z'
    if os.path.exists(DATAINDEX):
        df_idx = pd.read_csv(DATAINDEX)
        mask = (df_idx['filename'] == filename) & (df_idx['status'] == 'active')
        if mask.any():
            df_idx.loc[mask, 'status'] = 'superseded'
            df_idx.loc[mask, 'superseded_at'] = ts
    else:
        df_idx = pd.DataFrame(columns=['timestamp','filename','source_url','local_path',
                                        'data_type','rows','size_kb','status','superseded_at','note'])
    row = {'timestamp': ts, 'filename': filename, 'source_url': source_url,
           'local_path': local_path, 'data_type': data_type, 'rows': rows,
           'size_kb': round(size_kb,1) if size_kb else None,
           'status': status, 'superseded_at': '', 'note': note}
    pd.concat([df_idx, pd.DataFrame([row])], ignore_index=True).to_csv(DATAINDEX, index=False)

FORCE_RELOAD  = CFG.get('force_reload', {})

def needs_rebuild(filepath, min_rows, ds_key):
    if FORCE_RELOAD.get(ds_key, False): return True
    if not os.path.exists(filepath): return True
    try:
        n = sum(1 for _ in open(filepath)) - 1
        return n < min_rows
    except: return True
    return False

---

### 1.1 ENTSO-E Grenzflüsse CH (Download)

**Quelle:** ENTSO-E Transparency Platform — `query_crossborder_flows`
**Methode:** ENTSO-E API via `entsoe-py`, jahresweise mit Retry bei 503
**Inhalt:** Stündliche physikalische Flüsse CH↔DE/AT/IT/FR [MW]
**Zweck:** Netto-Export zeigt ob CH importiert oder exportiert.


In [ ]:
# ── Datensatz: ENTSO-E Grenzflüsse CH (Download) ──────────────────────────────
# Quelle  : ENTSO-E Transparency Platform — query_crossborder_flows
# Methode : ENTSO-E API via entsoe-py, jahresweise mit Retry bei 503
# Inhalt  : Stündliche physikalische Flüsse CH↔DE/AT/IT/FR [MW]
# Zweck   : Empirische Validierung des Business Case (Import = hohe Preise)
# Pfad    : data/raw/ch_crossborder_raw.csv

CROSS_FILE = os.path.join(DIR_RAW, 'ch_crossborder_raw.csv')
NEIGHBORS  = {'DE': ('CH','DE'), 'AT': ('CH','AT'), 'IT': ('CH','IT'), 'FR': ('CH','FR')}

# Jahresweise Lade-Hilfsfunktion mit 503-Retry (identisch zu NB01 DS1/DS2)
def _fetch_crossborder_year(client, from_c, to_c, year, max_retries=3, wait_s=20):
    """Lädt Grenzfluss eines Jahres mit Retry bei 503 Service Unavailable."""
    import time
    from requests.exceptions import HTTPError
    start = pd.Timestamp(f'{year}-01-01', tz='Europe/Zurich')
    end   = pd.Timestamp(f'{year}-12-31 23:00', tz='Europe/Zurich')
    for attempt in range(1, max_retries + 1):
        try:
            return client.query_crossborder_flows(from_c, to_c, start=start, end=end)
        except HTTPError as e:
            if '503' in str(e) and attempt < max_retries:
                print(f'  503 → Versuch {attempt}/{max_retries}, warte {wait_s}s...')
                time.sleep(wait_s)
            else:
                raise

# Start-/Endjahr aus config.json
import datetime
_start = CFG['daten']['start_year']
_end   = CFG['daten']['end_year']
START_YEAR = int(_start)
END_YEAR   = datetime.datetime.now().year if str(_end) == 'heute' else int(_end)
YEARS      = list(range(START_YEAR, END_YEAR + 1))

FORCE_RELOAD = CFG['force_reload']
def needs_download(path, min_kb, key):
    if FORCE_RELOAD.get(key, False): return True
    if not os.path.exists(path): return True
    return os.path.getsize(path) / 1024 < min_kb

if not needs_download(CROSS_FILE, 10, 'crossborder'):
    print(f'Grenzfluss-Daten vorhanden ({os.path.getsize(CROSS_FILE)/1024:.0f} KB) – kein Re-Download.')
    df_cross = pd.read_csv(CROSS_FILE)
else:
    try:
        import subprocess, sys
        subprocess.check_call([sys.executable,'-m','pip','install','entsoe-py','-q'])
        from entsoe import EntsoePandasClient

        # API-Key: Umgebungsvariable oder direkt eintragen (wie in NB01)
        ENTSOE_API_KEY = (os.environ.get('ENTSOE_API_KEY')
                         or CFG.get('api_keys', {}).get('entsoe', ''))
        if not ENTSOE_API_KEY:
            raise ValueError('ENTSOE_API_KEY nicht gesetzt.')

        client = EntsoePandasClient(api_key=ENTSOE_API_KEY)
        print(f'Lade CH Grenzflüsse {START_YEAR}–{END_YEAR} ({len(YEARS)} Jahre)...')

        flows = {}
        for neighbor, (from_c, to_c) in NEIGHBORS.items():
            try:
                parts = []
                for year in YEARS:
                    exp = _fetch_crossborder_year(client, from_c, to_c, year)
                    imp = _fetch_crossborder_year(client, to_c, from_c, year)
                    parts.append(exp.sub(imp, fill_value=0))
                flows[neighbor] = pd.concat(parts).sort_index()
                flows[neighbor] = flows[neighbor][~flows[neighbor].index.duplicated(keep='first')]
                print(f'  {neighbor}: {len(flows[neighbor]):,} h OK')
            except Exception as e:
                print(f'  {neighbor}: nicht verfügbar ({e})')

        if not flows:
            raise ValueError('Keine Grenzfluss-Daten verfügbar — alle Nachbarn fehlgeschlagen.')

        df_all = pd.DataFrame(flows)
        df_all.index = pd.to_datetime(df_all.index, utc=True)
        df_all['net_export_mw'] = df_all.sum(axis=1)
        df_all = df_all.reset_index().rename(columns={'index': 'timestamp'})
        df_all['timestamp'] = pd.to_datetime(df_all['timestamp'], utc=True)

        df_cross = df_all[['timestamp','net_export_mw'] +
                           [n for n in NEIGHBORS if n in df_all.columns]]
        df_cross.to_csv(CROSS_FILE, index=False, encoding='utf-8')
        kb = os.path.getsize(CROSS_FILE)/1024
        print(f'Gespeichert: {CROSS_FILE} | {len(df_cross):,} h | {kb:.0f} KB')

    except Exception as e:
        print(f'⚠️  Grenzflüsse nicht ladbar: {e}')
        print('→ Analyse wird übersprungen. FORCE_RELOAD[crossborder]=True zum Wiederholen.')
        df_cross = None


In [ ]:
# ── Verifikation: Grenzflüsse ────────────────────────────────────────────────
print(f'Shape   : {df_cross.shape}')
print(f'Zeitraum: {df_cross["timestamp"].min()} – {df_cross["timestamp"].max()}')
print(f'Nulls   : {df_cross.isnull().sum().sum()}')
print(f'Range   : {df_cross["net_export_mw"].min():.0f} – {df_cross["net_export_mw"].max():.0f} MW')
df_cross.head(3)


---

### 1.2 Import/Export-Analyse berechnen

**Zweck:** Verbindet `ch_crossborder_raw.csv` mit den bereinigten Spot-Preisen aus NB02.
Ergebnis: `ch_import_export_analyse.csv` in `data/intermediate/`.

**Voraussetzung:** `ch_spot_prices_clean.csv` muss vorhanden sein (NB02 Sektion 1).

> Diese Berechnung war früher in NB02 — sie gehört hier, da sie Kür-Daten (Grenzflüsse) benötigt.


In [ ]:
if df_cross is None:
    print('⚠️  Grenzflüsse nicht verfügbar — Import/Export-Analyse übersprungen.')
    df_cx = None
else:
    # ── Import/Export-Analyse berechnen → intermediate/ ──────────────────────────
    # Verbindet ch_crossborder_raw.csv mit bereinigten Spot-Preisen.
    # Ergebnis: ch_import_export_analyse.csv
    # Spalten: timestamp, net_export_mw, price_eur_mwh, dispatch
    
    CROSS_RAW  = os.path.join(DIR_RAW,   'ch_crossborder_raw.csv')
    CROSS_OUT  = os.path.join(DIR_INTER, 'ch_import_export_analyse.csv')
    CLEAN_FILE = os.path.join(DIR_PROCESSED, 'ch_spot_prices_clean.csv')
    
    if not os.path.exists(CROSS_RAW) or os.path.getsize(CROSS_RAW) < 10_000:
        print('ch_crossborder_raw.csv nicht vorhanden – Abschnitt übersprungen.')
        df_cx = None
    elif not os.path.exists(CLEAN_FILE):
        print('⚠️  ch_spot_prices_clean.csv fehlt → NB02 Sektion 1 ausführen.')
        df_cx = None
    elif not needs_rebuild(CROSS_OUT, 100, 'import_export'):
        print(f'ch_import_export_analyse.csv vorhanden ({os.path.getsize(CROSS_OUT)/1024:.0f} KB) – kein Rebuild.')
        df_cx = pd.read_csv(CROSS_OUT)
        df_cx['timestamp'] = pd.to_datetime(df_cx['timestamp'], utc=True)
    else:
        print('Berechne Import/Export-Analyse...')
    
        df_cross = pd.read_csv(CROSS_RAW)
        df_cross['timestamp'] = pd.to_datetime(df_cross['timestamp'], utc=True)
        df_cross = df_cross[['timestamp', 'net_export_mw']].set_index('timestamp')
    
        df_px = pd.read_csv(CLEAN_FILE)
        df_px['timestamp'] = pd.to_datetime(df_px['timestamp'], utc=True)
        df_px = df_px[['timestamp', 'price_eur_mwh']].set_index('timestamp')
    
        df_cx = df_cross.join(df_px, how='inner').reset_index()
        df_cx['timestamp'] = pd.to_datetime(df_cx['timestamp'], utc=True)
    
        # Dispatch-Flag: Preis >= p75 des Tages → Batterie würde einspeisen
        df_cx['date']     = df_cx['timestamp'].dt.date
        df_cx['p75_day']  = df_cx.groupby('date')['price_eur_mwh'].transform(
            lambda x: x.quantile(0.75))
        df_cx['dispatch'] = df_cx['price_eur_mwh'] >= df_cx['p75_day']
        df_cx = df_cx.drop(columns=['date', 'p75_day'])
    
        df_cx.to_csv(CROSS_OUT, index=False, encoding='utf-8')
        kb = os.path.getsize(CROSS_OUT) / 1024
        log_dataindex('ch_import_export_analyse.csv',
                      'NB07: join(crossborder_raw, spot_prices_clean)',
                      CROSS_OUT, 'intermediate',
                      rows=len(df_cx), size_kb=kb,
                      note='net_export_mw + price_eur_mwh + dispatch flag')
        imp = df_cx['net_export_mw'] < 0
        pi  = df_cx.loc[ imp, 'price_eur_mwh'].mean()
        pe  = df_cx.loc[~imp, 'price_eur_mwh'].mean()
        print(f'Gespeichert: {CROSS_OUT} | {len(df_cx):,} Zeilen | {kb:.0f} KB')
        print(f'  Import-Stunden : {imp.sum():,} ({imp.mean()*100:.1f}%)')
        print(f'  Ø Preis Import : {pi:.1f} EUR/MWh')
        print(f'  Ø Preis Export : {pe:.1f} EUR/MWh')
        result = 'bestätigt ✓' if pi > pe else 'nicht bestätigt ✗'
        print(f'  Business Case  : {result} (Import teurer: {pi-pe:+.1f} EUR/MWh)')
    

In [ ]:
# ── Verifikation: Import/Export-Analyse ──────────────────────────────────────
if df_cx is not None:
    print(f'Shape  : {df_cx.shape}')
    print(f'Spalten: {list(df_cx.columns)}')
    print(f'Nulls  : {df_cx.isnull().sum().sum()}')
    df_cx.head(3)
else:
    print('Import/Export-Analyse nicht verfügbar — Grenzflüsse fehlen.')


---
## 2. Analyse

### Kernthese
CH importiert Strom genau dann wenn die Preise hoch sind — der optimale  
Dispatch-Zeitpunkt für Batteriespeicher fällt damit mit den Nettoimport-Stunden zusammen.

| Kennzahl | Erwartung | Implikation |
|---------|----------|-------------|
| Ø Preis Import-Stunden > Ø Preis Export-Stunden | ✓ bestätigt | Business Case valide |
| Batterie-Dispatch in Import-Stunden > Zufall | ✓ bestätigt | Dispatch ist systemisch |


In [ ]:
if df_cx is None:
    print('⚠️  Übersprungen — Grenzflüsse nicht verfügbar.')
else:
    # ── Analyse: Import vs. Export Preisniveau ────────────────────────────────────
    df_cx['is_import']   = df_cx['net_export_mw'] < 0
    df_cx['hour']        = df_cx['timestamp'].dt.hour
    df_cx['month']       = df_cx['timestamp'].dt.month
    df_cx['season_name'] = df_cx['month'].map({12:'Winter',1:'Winter',2:'Winter',
                                                3:'Frühling',4:'Frühling',5:'Frühling',
                                                6:'Sommer',7:'Sommer',8:'Sommer',
                                                9:'Herbst',10:'Herbst',11:'Herbst'})
    
    # Stündliche Import-Häufigkeit
    h_imp = df_cx.groupby('hour')['is_import'].mean() * 100
    h_px  = df_cx.groupby('hour')['price_eur_mwh'].mean()
    
    print('Stunden mit höchster Import-Wahrscheinlichkeit:')
    print(h_imp.nlargest(5).round(1).to_string())
    print()
    print('Saisonale Analyse (Preis Import vs. Export):')
    for s in ['Winter','Frühling','Sommer','Herbst']:
        sub = df_cx[df_cx['season_name']==s]
        pi = sub.loc[sub['is_import'], 'price_eur_mwh'].mean()
        pe = sub.loc[~sub['is_import'], 'price_eur_mwh'].mean()
        print(f'  {s:<10}: Import {pi:.1f} vs. Export {pe:.1f} EUR/MWh  (Δ {pi-pe:+.1f})')
    

---
## 3. Visualisierung

### Chart 6: Import/Export-Analyse (4 Panels)
Diese Visualisierung gehört zur Kür-Analyse — Chart 6 des Projekts, berechnet in diesem Notebook.


In [ ]:
if df_cx is None:
    print('⚠️  Übersprungen — Grenzflüsse nicht verfügbar.')
else:
    # ── Chart CB-1: Import/Export vs. Preis (4 Panels) ───────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.patch.set_facecolor(BG_DARK)
    fig.suptitle('Cross-Border-Analyse: CH Import/Export & Preis-Koinzidenz',
                 color='white', fontsize=13, fontweight='bold')
    
    for ax in axes.flat:
        ax.set_facecolor(BG_PANEL)
        ax.tick_params(colors='#bbbbbb')
        for sp in ax.spines.values(): sp.set_edgecolor('#333333')
    
    # Panel 1: Stündliches Import-Profil
    ax = axes[0,0]
    ax.bar(h_imp.index, h_imp.values, color=['#EF5350' if v > 40 else '#FFA726' for v in h_imp.values], alpha=0.85)
    ax.set_title('Import-Wahrscheinlichkeit je Stunde', color='white')
    ax.set_xlabel('Stunde', color='#aaa'); ax.set_ylabel('Import-Häufigkeit [%]', color='#aaa')
    ax.grid(True, alpha=0.10)
    
    # Panel 2: Preisverteilung Import vs. Export
    ax = axes[0,1]
    imp_prices = df_cx.loc[df_cx['is_import'], 'price_eur_mwh']
    exp_prices = df_cx.loc[~df_cx['is_import'], 'price_eur_mwh']
    ax.hist(imp_prices, bins=60, alpha=0.7, color='#EF5350', label=f'Import (Ø {imp_prices.mean():.0f})', density=True)
    ax.hist(exp_prices, bins=60, alpha=0.7, color='#66BB6A', label=f'Export (Ø {exp_prices.mean():.0f})', density=True)
    ax.set_title('Preisverteilung: Import vs. Export', color='white')
    ax.set_xlabel('Preis [EUR/MWh]', color='#aaa')
    ax.legend(fontsize=9, framealpha=0.3, facecolor='#111', labelcolor='white')
    ax.grid(True, alpha=0.10)
    
    # Panel 3: Scatter Preis vs. Netto-Export
    ax = axes[1,0]
    sc = ax.scatter(df_cx['price_eur_mwh'], df_cx['net_export_mw'],
                    alpha=0.05, s=2, c=df_cx['hour'], cmap='plasma')
    ax.axhline(0, color='white', lw=1, alpha=0.5)
    ax.set_title('Preis vs. Netto-Export [MW]', color='white')
    ax.set_xlabel('Spot-Preis [EUR/MWh]', color='#aaa')
    ax.set_ylabel('Netto-Export [MW]', color='#aaa')
    ax.grid(True, alpha=0.10)
    
    # Panel 4: Saisonaler Preisvergleich Import vs. Export
    ax = axes[1,1]
    seasons = ['Winter','Frühling','Sommer','Herbst']
    x = range(len(seasons))
    pi_vals = [df_cx.loc[(df_cx['season_name']==s)&df_cx['is_import'],'price_eur_mwh'].mean() for s in seasons]
    pe_vals = [df_cx.loc[(df_cx['season_name']==s)&~df_cx['is_import'],'price_eur_mwh'].mean() for s in seasons]
    width = 0.35
    ax.bar([i - width/2 for i in x], pi_vals, width, label='Import', color='#EF5350', alpha=0.85)
    ax.bar([i + width/2 for i in x], pe_vals, width, label='Export', color='#66BB6A', alpha=0.85)
    ax.set_title('Saisonaler Preis Import vs. Export', color='white')
    ax.set_xticks(list(x)); ax.set_xticklabels(seasons, color='#aaa')
    ax.set_ylabel('Ø Preis [EUR/MWh]', color='#aaa')
    ax.legend(fontsize=9, framealpha=0.3, facecolor='#111', labelcolor='white')
    ax.grid(True, alpha=0.10)
    
    plt.tight_layout()
    out_path = os.path.join(CHARTS_DIR, 'kuer_nb07_import_export.png')
    plt.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
    plt.show(); plt.close()
    print(f'Gespeichert: {out_path}')
    

---
## 4. Fazit

**Kernergebnis:** CH importiert Strom in Hochpreis-Stunden — dies bestätigt empirisch  
den Business Case: Batteriespeicher einspeisen genau dann wenn das Netz es am meisten braucht.

| Erkenntnis | Detail |
|-----------|--------|
| Import teurer als Export | Δ aus Simulation (run NB07 mit Echtdaten) |
| Abendstunden (17–21h) | Höchste Import-Wahrscheinlichkeit |
| Winter | Grösste Preis-Kluft Import vs. Export |
| Dispatch-Koinzidenz | Batterie-Dispatch-Stunden decken sich mit Import-Stunden |

> Verweis: Chart 6 in NB03 zeigt dieselbe Analyse eingebettet im Pflicht-Bericht.  
> NB07 vertieft die Betrachtung um Saisonalität und Länderperspektive.

---
| [← NB06 Räumliche Analyse](06_Raeumliche_Analyse.ipynb) | [↑ Projektübersicht](00_Project_Overview.ipynb) | [→ NB08 Marktdynamik](08_Marktdynamik.ipynb) |
|:---|:---:|---:|
